# Chapter 1 — Pareto optimization of the affine Boy colormap in OKLab

We keep the display map itself in encoded sRGB,
\[
\boxed{\mathbf c_{\rm sRGB}(\mathbf n)=A\mathbf p_B(\mathbf n)+\mathbf b},
\]
and keep the hard display-gamut condition
\[
\boxed{0\le c_{{\rm sRGB},k}(\mathbf n)\le1\quad\forall \mathbf n\in S^2.}
\]

Perceptual utilities are evaluated after converting the sRGB color to OKLab.  In particular, the axis semantics are still
\[
\mathbf e_x\to {\rm red},\qquad \mathbf e_y\to {\rm green},\qquad \mathbf e_z\to {\rm blue}.
\]
The three targets are the OKLab coordinates of the exact sRGB primaries \((1,0,0),(0,1,0),(0,0,1)\), not arbitrary OKLab colors.

We trace the Pareto frontier by
\[
\min_{A,\mathbf b}J_{\rm axis}^{\rm OKLab}
\quad\text{s.t.}\quad
J_{\rm loc}^{\rm OKLab}\le t,
\quad
0\le A\mathbf p_B(\mathbf n)+\mathbf b\le1.
\]


## 1.1 Fixed Boy polynomial

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
np.set_printoptions(precision=8, suppress=True)

def boy_map(n):
    n = np.asarray(n, float)
    x, y, z = np.moveaxis(n, -1, 0)
    x2, y2, z2 = x*x, y*y, z*z
    p1 = 0.5*((2*x2-y2-z2) + 2*y*z*(y2-z2) + z*x*(x2-z2) + x*y*(y2-x2))
    p2 = (7/8)*((y2-z2) + z*x*(z2-x2) + x*y*(y2-x2))
    p3 = (1/8)*(x+y+z)*((x+y+z)**3 + 4*(y-x)*(z-y)*(x-z))
    return np.stack((p1, p2, p3), axis=-1)

def fibonacci_sphere(n):
    i = np.arange(n, dtype=float)
    z = 1 - 2*(i + 0.5)/n
    phi = np.pi*(3 - np.sqrt(5))*i
    r = np.sqrt(np.maximum(0, 1-z*z))
    return np.c_[r*np.cos(phi), r*np.sin(phi), z]

P_AXIS = boy_map(np.eye(3))

def pack(A, b):
    return np.r_[np.asarray(A).ravel(), np.asarray(b).ravel()]

def unpack(v):
    return v[:9].reshape(3, 3), v[9:12]

def axis_colors_srgb(A, b):
    return P_AXIS @ A.T + b


## 1.2 sRGB \(\rightarrow\) OKLab and the axis utility

The three semantic targets remain the display primaries.  We therefore define
\[
J_{\rm axis}^{\rm OKLab}
=
\sum_{\alpha=x,y,z}
\left\|
{\rm OKLab}\!\left(\mathbf c_{\rm sRGB}(\mathbf e_\alpha)\right)
-
{\rm OKLab}(\mathbf e_\alpha)
\right\|^2.
\]


In [ ]:
M1 = np.array([
    [0.4122214708, 0.5363325363, 0.0514459929],
    [0.2119034982, 0.6806995451, 0.1073969566],
    [0.0883024619, 0.2817188376, 0.6299787005],
])
M2 = np.array([
    [0.2104542553, 0.7936177850, -0.0040720468],
    [1.9779984951, -2.4285922050, 0.4505937099],
    [0.0259040371, 0.7827717662, -0.8086757660],
])

def srgb_to_linear(c):
    c = np.asarray(c, float)
    return np.where(c <= 0.04045, c/12.92, ((c + 0.055)/1.055)**2.4)

def srgb_to_oklab(c):
    linear = srgb_to_linear(c)
    lms = linear @ M1.T
    return np.cbrt(lms) @ M2.T

OKLAB_PRIMARY_TARGETS = srgb_to_oklab(np.eye(3))

def J_axis(v):
    A, b = unpack(v)
    c = axis_colors_srgb(A, b)
    # Gamut constraints keep these in [0,1]; clipping is only a numerical guard
    # while SLSQP is probing trial points.
    lab = srgb_to_oklab(np.clip(c, 0.0, 1.0))
    return np.sum((lab - OKLAB_PRIMARY_TARGETS)**2)


## 1.3 Local derivative uniformity in OKLab

Let
\[
f(\mathbf n)={\rm OKLab}(\mathbf c_{\rm sRGB}(\mathbf n)).
\]
The tangent derivative is
\[
D_T f
=
J_{{\rm OKLab}\leftarrow{\rm sRGB}}(\mathbf c)
\,A\,D\mathbf p_B\,P,
\qquad
P=I-\mathbf n\mathbf n^T.
\]
The induced tangent metric is \(G=(D_Tf)^T(D_Tf)\).  We use the same normalized uniformity functional,
\[
\boxed{
J_{\rm loc}^{\rm OKLab}
=
\frac{\langle {\rm tr}(G^2)\rangle}
{\langle {\rm tr}G\rangle^2}
-\frac12 .
}
\]
Unlike the previous sRGB version, the OKLab Jacobian depends on the current color, so this objective is evaluated numerically on a fixed deterministic sphere.


In [ ]:
def dlinear_dsrgb(c):
    c = np.asarray(c, float)
    return np.where(
        c <= 0.04045,
        1/12.92,
        (2.4/1.055)*((c + 0.055)/1.055)**1.4,
    )

def oklab_jacobian_srgb(c):
    c = np.asarray(c, float)
    linear = srgb_to_linear(c)
    lms = linear @ M1.T
    # For in-gamut nonnegative RGB, LMS is nonnegative.  The small floor only
    # regularizes the derivative at the black corner.
    dcuberoot = (1/3)*np.maximum(lms, 1e-14)**(-2/3)
    dlin = dlinear_dsrgb(c)
    return np.einsum("ab,...b,bc,...c->...ac", M2, dcuberoot, M1, dlin)

# Fixed deterministic quadrature grid for the OKLab local utility.
N_LOCAL = 1200
N_LOCAL_DIR = fibonacci_sphere(N_LOCAL)
P_LOCAL = boy_map(N_LOCAL_DIR)
PROJ_LOCAL = (
    np.eye(3)[None, :, :]
    - N_LOCAL_DIR[:, :, None]*N_LOCAL_DIR[:, None, :]
)

# Dp_B is precomputed once.  A centered difference here only differentiates the
# fixed polynomial map; it is not part of the optimizer's finite differences.
_eps = 2e-6
B_LOCAL = np.empty((N_LOCAL, 3, 3))
for j in range(3):
    d = np.zeros(3)
    d[j] = _eps
    B_LOCAL[:, :, j] = (boy_map(N_LOCAL_DIR + d) - boy_map(N_LOCAL_DIR - d))/(2*_eps)

def J_local(v):
    A, b = unpack(v)
    c = P_LOCAL @ A.T + b
    c = np.clip(c, 0.0, 1.0)
    Jc = oklab_jacobian_srgb(c)
    D = np.einsum("nij,jk,nkl->nil", Jc, A, B_LOCAL)
    DT = np.einsum("nij,njk->nik", D, PROJ_LOCAL)
    G = np.einsum("nji,njk->nik", DT, DT)
    trG = np.trace(G, axis1=1, axis2=2)
    trG2 = np.einsum("nij,nji->n", G, G)
    mean_tr = trG.mean()
    if mean_tr <= 1e-14:
        return np.inf
    return trG2.mean()/(mean_tr*mean_tr) - 0.5


## 1.4 Hard sRGB gamut and deterministic constraint generation

The perceptual objective is in OKLab, but feasibility remains exactly in encoded sRGB:
\[
0\le A\mathbf p_B(\mathbf n)+\mathbf b\le1.
\]
Thus the gamut constraint keeps the same simple channel-wise form as before.


In [ ]:
def gamut_constraints(v, P):
    A, b = unpack(v)
    C = P @ A.T + b
    return np.r_[C.ravel(), (1-C).ravel()]

def solve_point(
    t,
    v0,
    n_initial=3500,
    n_verify=100000,
    max_rounds=10,
    tol=2e-7,
):
    Pcon = boy_map(fibonacci_sphere(n_initial))
    Pver = boy_map(fibonacci_sphere(n_verify))
    v = v0.copy()

    for iround in range(max_rounds):
        cons = [
            {"type": "ineq", "fun": lambda x, tt=t: tt - J_local(x)},
            {"type": "ineq", "fun": lambda x, PP=Pcon: gamut_constraints(x, PP)},
        ]
        res = minimize(
            J_axis,
            v,
            method="SLSQP",
            constraints=cons,
            options={"ftol": 5e-10, "maxiter": 900, "disp": False},
        )
        v = res.x
        A, b = unpack(v)
        C = Pver @ A.T + b
        mins, maxs = C.min(0), C.max(0)

        idx = []
        for k in range(3):
            idx += [np.argmin(C[:, k]), np.argmax(C[:, k])]
        Pcon = np.vstack((Pcon, Pver[np.unique(idx)]))

        if mins.min() >= -tol and maxs.max() <= 1 + tol:
            break

    A, b = unpack(v)
    return dict(
        t=t, v=v, A=A, b=b,
        J_axis=J_axis(v), J_local=J_local(v),
        axis=axis_colors_srgb(A, b),
        gmin=mins, gmax=maxs,
        rounds=iround + 1,
        success=res.success, message=res.message,
    )


## 1.5 Trace the OKLab Pareto frontier

The numerical scale of \(J_{\rm loc}^{\rm OKLab}\) is different from the old sRGB objective, so the old threshold values should not be reused.  The following range spans the useful tradeoff found in initial tests.


In [ ]:
# Previous sRGB Pareto solution, used only as a deterministic starting point.
A0 = np.array([
    [ 0.52868237,  0.05496927,  0.19053080],
    [-0.31232790,  0.42594150,  0.18920397],
    [-0.21649449, -0.48049833,  0.19067196],
])
b0 = np.array([0.39350715, 0.39366654, 0.39346801])
v = pack(A0, b0)

T = [0.76, 0.70, 0.65, 0.60, 0.55, 0.50, 0.45, 0.40, 0.35, 0.30]
pareto = []
for t in T:
    r = solve_point(t, v)
    pareto.append(r)
    v = r["v"]
    print(
        f"t={t:.3f}  J_axis_OKLab={r['J_axis']:.6f}  "
        f"J_local_OKLab={r['J_local']:.6f}  rounds={r['rounds']}"
    )


In [ ]:
rows = []
for r in pareto:
    C = r["axis"]
    rows.append(dict(
        t=r["t"],
        J_local_OKLab=r["J_local"],
        J_axis_OKLab=r["J_axis"],
        x_R=C[0,0], x_G=C[0,1], x_B=C[0,2],
        y_R=C[1,0], y_G=C[1,1], y_B=C[1,2],
        z_R=C[2,0], z_G=C[2,1], z_B=C[2,2],
        min_gamut=np.min(r["gmin"]),
        max_gamut=np.max(r["gmax"]),
    ))

df = pd.DataFrame(rows)
display(df)

plt.figure(figsize=(6,4))
plt.plot(df.J_local_OKLab, df.J_axis_OKLab, marker="o")
plt.xlabel(r"$J_{\rm loc}^{\rm OKLab}$")
plt.ylabel(r"$J_{\rm axis}^{\rm OKLab}$")
plt.grid(True, alpha=.25)
plt.show()

for target in (0.70, 0.60, 0.50, 0.40, 0.30):
    r = min(pareto, key=lambda x: abs(x["t"]-target))
    print("\n" + "="*64)
    print("t =", r["t"])
    print("J_local_OKLab =", r["J_local"])
    print("J_axis_OKLab  =", r["J_axis"])
    print("A =\n", r["A"])
    print("b =", r["b"])
    print("axis colors in sRGB =\n", r["axis"])
    print("gamut min =", r["gmin"], "gamut max =", r["gmax"])
